# Image Processing Pipeline


Import libraries and packages + installations



In [ ]:
# Upgrade pip and install packages
!pip install --upgrade pip
!pip install --quiet torchio hd-bet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metad

In [ ]:
import torchio as tio
import SimpleITK as sitk
import torch
import subprocess
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import numpy as np
from multiprocessing import Pool, cpu_count
from threading import Thread
import shutil
from queue import Queue
import HD_BET

Upload Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


GPU Set-Up

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


Process Subject Files

In [ ]:
def process_file(local_input, local_skull, clean_output):
    # Run skull-stripping
    image = run_hd_bet(local_input, local_skull)

    # Resample image
    image = image_resample(image)

    # Normalize and save final image
    normalization(image, clean_output)

    # Cleanup temporary files
    os.remove(local_input)
    os.remove(local_skull)

Bias Correction Function

In [ ]:
def bias_correction(input_path, output_path): # Bias Field Correction (N4ITK algorithm)
    print("Running Bias Correction")
    input_img = sitk.ReadImage(input_path, sitk.sitkFloat32)
    mask_img = sitk.OtsuThreshold(input_img, 0, 1, 200)
    shrinkFactor = 1
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    numberFittingLevels = 4
    corrected_image = corrector.Execute(input_img, mask_img)
    log_bias_field = corrector.GetLogBiasFieldAsImage(input_img)
    corrected_image_fr = corrected_image / sitk.Exp(log_bias_field)
    sitk.WriteImage(corrected_image_fr, output_path)

Skull Stripping Functions

In [ ]:
def run_synthstrip(input_path, output_path):
    if os.path.exists(output_path):
        return
    print("Running Skull Stripping")
    input_path = os.path.abspath(input_path)
    # Fix: Ensure the model path matches the actual path on Google Drive (with underscore)
    model_path = "/content/local_models/synthstrip.1.pt"

    cmd = [
        "nipreps-synthstrip",
        "-i", input_path,
        "-o", output_path,
        "--model", model_path,
        "--gpu"
    ]
    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print("Synthstrip STDOUT:", result.stdout)
        print("Synthstrip STDERR:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Synthstrip command failed with exit code {e.returncode}")
        print(f"Synthstrip STDOUT:\n{e.stdout}")
        print(f"Synthstrip STDERR:\n{e.stderr}")
        raise e

    image = tio.ScalarImage(output_path)
    return image

In [ ]:
def run_hd_bet(input_path, output_path, device="cuda"):
    if os.path.exists(output_path):
        print(f"File already exists: {output_path}. Loading existing image.")
        return tio.ScalarImage(output_path)

    print(f"Running HD-BET Skull Stripping on: {input_path}")
    input_path = os.path.abspath(input_path)
    output_path = os.path.abspath(output_path)

    cmd = ["hd-bet", "-i", input_path, "-o", output_path, "-device", "cuda", "--disable_tta",]
    print(f"HD-BET command: {' '.join(cmd)}")

    result = None
    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print("HD-BET STDOUT:", result.stdout)
        print("HD-BET STDERR:", result.stderr)
        print("Return code:", result.returncode)
    except subprocess.CalledProcessError as e:
        print(f"\n❌ HD-BET FAILED for {input_path}")
        print("Return code:", e.returncode)
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)
        return None

    if not os.path.exists(output_path):
        print(f"ERROR: HD-BET command completed but output file not found at: {output_path}")

        if output_path.endswith(".nii.gz"):
            stem = output_path[: -len(".nii.gz")]
            potential_bet_output_path = f"{stem}_bet.nii.gz"
        else:
            stem, ext = os.path.splitext(output_path)
            potential_bet_output_path = f"{stem}_bet{ext}"

        stdout_txt = result.stdout if result else "N/A"
        stderr_txt = result.stderr if result else "N/A"

        if os.path.exists(potential_bet_output_path):
            print(f"Found potential HD-BET output at: {potential_bet_output_path}. Using this path.")
            output_path = potential_bet_output_path
        else:
            raise FileNotFoundError(
                f'HD-BET output file not found at expected location: "{output_path}" '
                f'and not at potential _bet location: "{potential_bet_output_path}".\n'
                f'HD-BET STDOUT:\n{stdout_txt}\n'
                f'HD-BET STDERR:\n{stderr_txt}'
            )

    return tio.ScalarImage(output_path)

Image Resizing Function

In [ ]:
def image_resample(image):
   if image == None:
    return
   print('Running Image Resample')
   resize = tio.Resize((128, 128, 128))
   resampled_image = resize(image)
   return resampled_image

Normalization Function


In [ ]:
def normalization(image, output_path):
    if image == None:
      return
    print('Running Normalization')
    transform = tio.ZNormalization()
    normalized = transform(image)
    normalized.save(output_path)

In [ ]:
def dwt_compression(image):
  wavelet_type = 'db2'
  level = 1
  coeffs = pywt.wavedecn(image, wavelet=wavelet_type, level=level, mode='symmetric')
  compressed_3d_array = coeffs[0]

  print(f"Original shape: {image.shape}")
  print(f"Compressed shape: {compressed_3d_array.shape}")
  output = compressed_3d_array[:91,:109,:91]
  return output

# Data Preparation

In [ ]:
class MRIDataset(Dataset):
    def __init__(self, subjects, modalities=['t1c', 't1n', 't2f', 't2w'], transform=None):
        """
        subjects: list of folder paths for each subject
        modalities: list of 4 MRI modalities
        transform: optional augmentation / preprocessing
        """
        self.subjects = subjects
        self.modalities = modalities
        self.transform = transform

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        subj_path = self.subjects[idx]

        # Load modalities
        imgs = []
        for mod in self.modalities:
            img_path = os.path.join(subj_path, f"{mod}.nii.gz")
            img = nib.load(img_path).get_fdata().astype(np.float32)
            imgs.append(img)
        x = np.stack(imgs, axis=0)  # shape: [4, D, H, W]

        # Load segmentation mask
        mask_path = os.path.join(subj_path, "seg.nii.gz")
        y = nib.load(mask_path).get_fdata().astype(np.float32)  # binary mask

        # Optional: preprocessing / augmentation
        if self.transform:
            x, y = self.transform(x, y)

        return torch.tensor(x), torch.tensor(y).unsqueeze(0)  # [C,D,H,W], [1,D,H,W]

In [ ]:
# Get all subject folders
dataset_dir = "/content/drive/MyDrive/ECE1508_PROJECT/clean_dataset"
all_subjects = [os.path.join(dataset_dir, s) for s in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, s))]

# Split 70% train, 15% val, 15% test
train_subjects, test_subjects = train_test_split(all_subjects, test_size=0.3, random_state=42)
val_subjects, test_subjects = train_test_split(test_subjects, test_size=0.5, random_state=42)

print("Train:", len(train_subjects), "Val:", len(val_subjects), "Test:", len(test_subjects))

ValueError: With n_samples=0, test_size=0.3 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
batch_size = 5  # adjust depending on GPU memory

train_dataset = MRIDataset(train_subjects)
val_dataset = MRIDataset(val_subjects)
test_dataset = MRIDataset(test_subjects)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

NameError: name 'MRIDataset' is not defined

# Machine Learning Pipeline


Model Architecture

In [ ]:
class UNet3D(nn.Module):
    def __init__(self, in_channels=4, out_channels=1, features=[32, 64, 128]):
        super(UNet3D, self).__init__()

        # Encoder
        self.encoders = nn.ModuleList()
        for f in features:
            self.encoders.append(self.conv_block(in_channels, f))
            in_channels = f

        # Pooling
        self.pool = nn.MaxPool3d(2)

        # Bottleneck
        self.bottleneck = self.conv_block(features[-1], features[-1]*2)

        # Decoder
        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        rev_features = features[::-1]
        for f in rev_features:
            self.upconvs.append(nn.ConvTranspose3d(features[-1]*2, f, kernel_size=2, stride=2))
            self.decoders.append(self.conv_block(f*2, f))
            features[-1] = f  # keep feature count consistent

        # Final convolution
        self.final_conv = nn.Conv3d(features[0], out_channels, kernel_size=1)

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        skip_connections = []

        # Encoder
        for enc in self.encoders:
            x = enc(x)
            skip_connections.append(x)
            x = self.pool(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder
        skip_connections = skip_connections[::-1]
        for idx in range(len(self.decoders)):
            x = self.upconvs[idx](x)
            skip = skip_connections[idx]

            # Pad if needed
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)

            x = torch.cat([skip, x], dim=1)
            x = self.decoders[idx](x)

        return self.final_conv(x)

Training

In [ ]:
def train(model, loss_function, num_epochs):
    optimizer = optim.Adam(model.parameters(), lr = 0.001, eps=1e-4)

    for epoch in range(num_epochs):
        train_loss = 0.0
        train_accuracy = 0.0

        # training loop
        for i, data in enumerate(trainloader):
            # reset initial states
            model.h_state = model.initial_state()

            # get the inputs
            inputs, labels = data

            # reshape input
            inputs = inputs.squeeze(1)

            # forward pass
            optimizer.zero_grad()
            outputs = model.forward(inputs);

            # compute loss
            loss = loss_function(outputs, labels);

            # backward pass
            loss.backward()
            optimizer.step()

            # update training loss and accuracy
            train_loss += loss.item();
            train_dice_score += dice_score(outputs, labels)

        print('Epoch:%d|Loss:%.4f|Dice:%.2f'
        %(epoch+1, train_loss / len(trainloader), train_dice_score / len(trainloader)))
    return

Test Function

In [ ]:
def test(model: UNet3D, loss_function, device):
  model = model.to(device = device);
  with torch.no_grad():
    risk = 0;
    accuracy = 0;

    model.eval();
    # loop over test mini-batches
    for i, (images, labels) in enumerate(test_loader):
      # reshape labels to have the same form as output
      # make sure labels are of torch.float32 type
      labels = labels.unsqueeze(1).float();
      # move tensors to the configured device
      images = images.to(device = device);
      labels = labels.to(device = device);
      # forward pass
      outputs = model(images);
      loss = loss_function(outputs, labels);
      # determine the class of output from sigmoid output
      predicted = (outputs >= 0.5).int();

      # compute the fraction of correctly predicted labels
      correct_predict = (predicted == labels).sum().item() / len(label)
      risk += loss.item();
      accuracy += correct_predict;

    # average test risk and accuracy over the whole test dataset
    test_risk = risk / len(test_loader);
    test_accuracy = dice_score * 100 / len(test_loader);
    return test_risk, test_accuracy

Dice Score Function

In [ ]:
def dice_score(z_out, labels, threshold=0.5, smooth=1e-6):
    # Apply sigmoid to get probabilities
    probs = torch.sigmoid(z_out)

    # Binarize predictions
    preds = (probs > threshold).float()

    # Flatten tensors
    preds_flat = preds.view(preds.size(0), -1)
    labels_flat = labels.view(labels.size(0), -1).float()

    # Compute intersection and union
    intersection = (preds_flat * labels_flat).sum(dim=1)
    union = preds_flat.sum(dim=1) + labels_flat.sum(dim=1)

    # Compute Dice score per batch element
    dice = (2.0 * intersection + smooth) / (union + smooth)

    # Return mean Dice over batch
    return dice.mean().item() * 100.0  # percentage

Dice Loss Function

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    dice = (2 * intersection + smooth) / (union + smooth)
    return 1 - dice

Main Function

In [ ]:
def ensure_dir(folder_path):
    """Ensure the path exists and is a directory."""
    if os.path.exists(folder_path):
        if not os.path.isdir(folder_path):
            os.remove(folder_path)
    os.makedirs(folder_path, exist_ok=True)

In [ ]:
def all_files_valid(folder, expected_count):
    """Check if folder has enough non-empty files."""
    files = os.listdir(folder)
    if len(files) < expected_count:
        return False
    return all(os.path.getsize(os.path.join(folder, f)) > 0 for f in files)

In [ ]:
import numpy as np
import pywt
import nibabel as nib

def dwt_compress_3d(image: np.ndarray, wavelet='db2', level=3, threshold_ratio=0.1):
    """
    True 3D DWT compression — operates across all 3 axes simultaneously.

    Args:
        image:           3D numpy array (X, Y, Z)
        wavelet:         wavelet type e.g. 'db2', 'haar', 'sym4'
        level:           decomposition level
        threshold_ratio: fraction of max detail coeff to zero out

    Returns:
        compressed 3D numpy array, same shape as input
    """
    original_shape = image.shape
    image = image.astype(np.float32)

    # --- 1. Forward 3D DWT ---
    coeffs = pywt.dwtn(image, wavelet=wavelet)
    # coeffs is a dict: keys like 'aaa', 'aad', 'ada', 'add', 'daa', 'dad', 'dda', 'ddd'
    # 'a' = approximation axis, 'd' = detail axis
    # 'aaa' is the low-freq approximation — keep it untouched

    # --- 2. Threshold all detail subbands ---
    # Compute global threshold from all detail coefficients
    detail_vals = np.concatenate([
        np.abs(v).ravel() for k, v in coeffs.items() if k != 'a' * image.ndim
    ])
    threshold = threshold_ratio * detail_vals.max()

    coeffs_thresh = {}
    approx_key = 'a' * image.ndim  # 'aaa' for 3D
    for key, subband in coeffs.items():
        if key == approx_key:
            coeffs_thresh[key] = subband  # never threshold approximation
        else:
            coeffs_thresh[key] = pywt.threshold(subband, value=threshold, mode='soft')

    # --- 3. Inverse 3D DWT ---
    reconstructed = pywt.idwtn(coeffs_thresh, wavelet=wavelet)

    # idwtn may pad slightly — crop back to original shape
    reconstructed = reconstructed[
        :original_shape[0],
        :original_shape[1],
        :original_shape[2]
    ]

    reconstructed = np.clip(reconstructed, image.min(), image.max())
    return reconstructed.astype(np.float32)


def dwt_compress_3d_multilevel(image: np.ndarray, wavelet='db2', level=3, threshold_ratio=0.1):
    """
    Multi-level 3D DWT — repeatedly decomposes the approximation subband.
    Better compression than single-level at the cost of slightly more compute.
    """
    original_shape = image.shape
    current = image.astype(np.float32)
    all_details = []  # store detail subbands at each level

    # --- 1. Forward multi-level 3D DWT ---
    approx_key = 'a' * image.ndim
    for _ in range(level):
        coeffs = pywt.dwtn(current, wavelet=wavelet)
        details = {k: v for k, v in coeffs.items() if k != approx_key}
        current = coeffs[approx_key]  # drill into approximation
        all_details.append(details)

    # --- 2. Global threshold across ALL detail subbands at ALL levels ---
    all_vals = np.concatenate([
        np.abs(v).ravel()
        for details in all_details
        for v in details.values()
    ])
    threshold = threshold_ratio * all_vals.max()

    thresholded_details = [
        {k: pywt.threshold(v, value=threshold, mode='soft') for k, v in details.items()}
        for details in all_details
    ]

    # --- 3. Inverse multi-level 3D DWT ---
    reconstructed = current  # start from deepest approximation
    for details in reversed(thresholded_details):
        coeffs_recon = {approx_key: reconstructed, **details}
        reconstructed = pywt.idwtn(coeffs_recon, wavelet=wavelet)
        # trim padding after each level
        reconstructed = reconstructed[
            :original_shape[0],
            :original_shape[1],
            :original_shape[2]
        ]

    reconstructed = np.clip(reconstructed, image.min(), image.max())
    return reconstructed.astype(np.float32)


def compression_stats(original: np.ndarray, compressed: np.ndarray, wavelet='db2', threshold_ratio=0.1):
    """Print useful compression diagnostics."""
    approx_key = 'a' * original.ndim
    coeffs = pywt.dwtn(original.astype(np.float32), wavelet=wavelet)

    total_coeffs = sum(v.size for v in coeffs.values())
    detail_vals = np.concatenate([np.abs(v).ravel() for k, v in coeffs.items() if k != approx_key])
    threshold = threshold_ratio * detail_vals.max()
    nonzero = coeffs[approx_key].size + np.sum(np.abs(detail_vals) > threshold)

    psnr = 10 * np.log10(original.max()**2 / np.mean((original - compressed)**2))

    print(f"Original shape:      {original.shape}")
    print(f"Compression ratio:   {total_coeffs / nonzero:.2f}x")
    print(f"Sparsity:            {100*(1 - nonzero/total_coeffs):.1f}% coefficients zeroed")
    print(f"PSNR:                {psnr:.2f} dB  ({'good' if psnr > 40 else 'ok' if psnr > 30 else 'lossy'})")
    print(f"Max error:           {np.abs(original - compressed).max():.4f}")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Thread-safe print
_print_lock = threading.Lock()
def tprint(*args, **kwargs):
    with _print_lock:
        print(*args, **kwargs)

def process_file(file, subject_folder, output_folder, tmp_dir):
    """Processes a single file: skull strip + resample + normalize."""
    input_path  = os.path.join(subject_folder, file)
    output_path = os.path.join(output_folder, file)

    # Segmentation masks: just copy
    if file.endswith("-seg.nii.gz"):
        shutil.copy(input_path, output_path)
        tprint(f"  [COPY] {file}")
        return file, True

    tmp_input  = os.path.join(tmp_dir, f"{threading.get_ident()}_{file}")
    tmp_output = os.path.join(tmp_dir, f"{threading.get_ident()}_skull_{file}")

    try:
        shutil.copy(input_path, tmp_input)          # Drive → local
        image = run_hd_bet(tmp_input, tmp_output)   # HD-BET
        if image is None:
            tprint(f"  [SKIP] HD-BET failed: {file}")
            return file, False
        image = image_resample(image)
        normalization(image, output_path)
        tprint(f"  [DONE] {file}")
        return file, True

    except Exception as e:
        tprint(f"  [ERR]  {file}: {e}")
        return file, False

    finally:
        for tmp in [tmp_input, tmp_output]:
            if os.path.exists(tmp):
                os.remove(tmp)


if __name__ == "__main__":
    directory = "/content/drive/MyDrive/ECE1508_PROJECT/dataset"
    base_out  = "/content/drive/MyDrive/ECE1508_PROJECT/clean_dataset"
    TMP_DIR   = "/content/tmp"
    os.makedirs(TMP_DIR, exist_ok=True)

    EXPECTED_FILES = 5
    MAX_WORKERS    = 4   # ← tune this (see note below)

    p        = Path(directory)
    subjects = sorted(os.listdir(directory))   # sorted for reproducible logs

    for subject in subjects:
        subject_folder = str(p / subject)
        output_folder  = os.path.join(base_out, subject)

        if not os.path.isdir(subject_folder):
            print(f"[SKIP] Not a directory: {subject_folder}")
            continue

        ensure_dir(output_folder)

        if all_files_valid(output_folder, EXPECTED_FILES):
            print(f"[SKIP] Already complete: {subject}")
            continue

        existing_files    = set(os.listdir(output_folder))
        files_to_process  = list(set(os.listdir(subject_folder)) - existing_files)

        print(f"\n{'='*50}")
        print(f"Subject: {subject} | Files to process: {len(files_to_process)}")

        # --- parallel file processing per subject ---
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(process_file, f, subject_folder, output_folder, TMP_DIR): f
                for f in files_to_process
            }
            for future in as_completed(futures):
                fname, ok = future.result()   # exceptions are re-raised here if any slipped through


Subject: BraTS-GLI-00005-100 | Files to process: 5
  [COPY] BraTS-GLI-00005-100-seg.nii.gz
Running HD-BET Skull Stripping on: /content/tmp/137039738873408_BraTS-GLI-00005-100-t2w.nii.gz
HD-BET command: hd-bet -i /content/tmp/137039738873408_BraTS-GLI-00005-100-t2w.nii.gz -o /content/tmp/137039738873408_skull_BraTS-GLI-00005-100-t2w.nii.gz -device cuda --disable_tta
Running HD-BET Skull Stripping on: /content/tmp/137039603291712_BraTS-GLI-00005-100-t1c.nii.gz
HD-BET command: hd-bet -i /content/tmp/137039603291712_BraTS-GLI-00005-100-t1c.nii.gz -o /content/tmp/137039603291712_skull_BraTS-GLI-00005-100-t1c.nii.gz -device cuda --disable_tta
Running HD-BET Skull Stripping on: /content/tmp/137039620077120_BraTS-GLI-00005-100-t2f.nii.gz
HD-BET command: hd-bet -i /content/tmp/137039620077120_BraTS-GLI-00005-100-t2f.nii.gz -o /content/tmp/137039620077120_skull_BraTS-GLI-00005-100-t2f.nii.gz -device cuda --disable_tta

❌ HD-BET FAILED for /content/tmp/137039620077120_BraTS-GLI-00005-100-t2f.nii


KeyboardInterrupt



In [ ]:
import SimpleITK as sitk
if __name__ == "__main__":
    directory = "/content/drive/MyDrive/ECE1508_PROJECT/dataset"
    path_1 = os.path.join(directory, 'BraTS-GLI-03064-100/BraTS-GLI-03064-100-t2w.nii.gz')
    image = sitk.ReadImage(path_1)
    np_array = sitk.GetArrayFromImage(image).astype(np.float32)
    tensor = torch.from_numpy(np_array).unsqueeze(0).to(torch.float32)
    output = dwt_compression(tensor)
    print("Tensor")
    print(tensor)
    print("Output")
    print(output)


/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 1 is too high: all coefficients will experience boundary effects.
  warnings.warn(


Original shape: torch.Size([1, 182, 218, 182])
Compressed shape: (2, 92, 110, 92)
Tensor
tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         ...,

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
     